# Clase 043 — SQL desde Python

**Parte 0** · sqlite3 + SQLAlchemy + DuckDB.

> 🎯 Las 3 formas que vas a usar en producción. Parametrización para evitar injection.

> ⏱️ ~75 min

## ⚙️ Setup

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
rng = np.random.default_rng(42)

# DataFrame demo
df = pd.DataFrame({
    'cliente_id': range(1, 11),
    'nombre': [f'Cliente {i}' for i in range(1, 11)],
    'pais': rng.choice(['ES', 'CL', 'MX'], 10),
    'monto': rng.uniform(50, 500, 10).round(2),
})
print(df.head())

## 1️⃣ `sqlite3` stdlib

Flow básico: `connect` → `cursor` → `execute(sql, params)` → `fetchall()`.

In [ ]:
con = sqlite3.connect(':memory:')
cur = con.cursor()
cur.execute('CREATE TABLE clientes (id INTEGER, nombre TEXT, pais TEXT, monto REAL)')

# executemany con tuples para insertar varios
datos = [(row.cliente_id, row.nombre, row.pais, row.monto) for row in df.itertuples()]
cur.executemany('INSERT INTO clientes VALUES (?, ?, ?, ?)', datos)
con.commit()

# Consulta con placeholder ?
for row in cur.execute('SELECT * FROM clientes WHERE pais = ? AND monto > ?', ('ES', 200)):
    print(row)

## 2️⃣ ⚠️ NUNCA concatenes SQL

```python
# ❌ MAL — vulnerable a injection
user_input = "ES'; DROP TABLE clientes; --"
cur.execute(f"SELECT * FROM clientes WHERE pais = '{user_input}'")  # ¡catástrofe!

# ✅ BIEN — placeholder seguro
cur.execute('SELECT * FROM clientes WHERE pais = ?', (user_input,))
```

El driver escapa el valor automáticamente. Es **la** regla de seguridad de SQL desde código.

## 3️⃣ `pd.read_sql` y `df.to_sql`

Pandas tiene pasarela bidireccional:

In [ ]:
# DataFrame → tabla
df.to_sql('clientes_pd', con, if_exists='replace', index=False)

# Tabla → DataFrame
result = pd.read_sql('SELECT pais, AVG(monto) AS avg_m FROM clientes_pd GROUP BY pais', con)
print(result)

## 4️⃣ SQLAlchemy — backend-agnostic

```python
from sqlalchemy import create_engine

# URLs por motor:
#   sqlite:///archivo.db           — SQLite local
#   sqlite:///:memory:             — SQLite en memoria
#   postgresql://user:pw@host/db   — Postgres
#   mysql+pymysql://user:pw@host/db— MySQL

engine = create_engine('sqlite:///:memory:')
df.to_sql('clientes', engine, if_exists='replace', index=False)
result = pd.read_sql('SELECT * FROM clientes WHERE monto > 200', engine)
```

Ventaja: cambias 1 string en el engine y migras de SQLite a Postgres sin tocar el resto del código.

In [ ]:
try:
    from sqlalchemy import create_engine
    engine = create_engine('sqlite:///:memory:')
    df.to_sql('cl', engine, if_exists='replace', index=False)
    print(pd.read_sql('SELECT pais, COUNT(*) c FROM cl GROUP BY pais', engine))
except ImportError:
    print('Instala SQLAlchemy: pip install sqlalchemy')

## 5️⃣ DuckDB — SQL sobre DataFrames y archivos

DuckDB es como SQLite pero **columnar** (optimizado para analytics) y **lee CSV/Parquet directamente** sin cargar a memoria:

```python
import duckdb
# Sobre DataFrame en memoria
duckdb.query('SELECT species, AVG(body_mass_g) FROM df GROUP BY species').df()

# Sobre CSV directo (sin pandas)
duckdb.query("SELECT species, COUNT(*) FROM 'penguins.csv' GROUP BY species").df()

# Sobre Parquet (mucho más rápido)
duckdb.query("SELECT * FROM 'datos.parquet' LIMIT 100").df()
```

In [ ]:
try:
    import duckdb
    # SQL sobre nuestro DataFrame
    result = duckdb.query('''
        SELECT pais,
               COUNT(*)       AS n,
               AVG(monto)     AS avg_monto,
               MAX(monto)     AS max_monto
        FROM df
        GROUP BY pais
        ORDER BY avg_monto DESC
    ''').df()
    print(result.round(2))
except ImportError:
    print('Instala DuckDB: pip install duckdb')

## 🧭 Cuándo cada uno

| Tool | Caso |
|---|---|
| `sqlite3` stdlib | Demos, tests, BDs locales pequeñas, scripts one-shot |
| SQLAlchemy | Producción con PostgreSQL/MySQL; ORMs, migraciones |
| DuckDB | Análisis ad-hoc sobre CSV/Parquet, EDA rápido con SQL |

## ✅ Checklist

- [ ] Uso placeholders `?` en sqlite3 (NUNCA concatenar)
- [ ] Sé pasarela `df.to_sql` / `pd.read_sql`
- [ ] Conozco SQLAlchemy para producción
- [ ] Uso DuckDB para SQL sobre CSV/Parquet sin cargar a pandas
- [ ] Sé qué tool elegir según contexto

## 📝 Homework

Ver `README.md`. 3 backends mismo análisis + demo injection.

## 🔗 Referencias

- [sqlite3](https://docs.python.org/3/library/sqlite3.html)
- [SQLAlchemy](https://docs.sqlalchemy.org/en/20/tutorial/)
- [DuckDB Python](https://duckdb.org/docs/api/python/overview)

➡️ **Siguiente:** [044 — MongoDB](../044-nosql-mongodb-con-pymongo/README.md)